# Deep Learning for Medical Image Classification: Tumor Detection Tutorial

## Overview

This comprehensive tutorial demonstrates how to build a complete deep learning pipeline for binary classification of medical images. We'll focus on classifying tumor images as **benign** or **malignant**, simulating the type of analysis done with MRI scans.

### What You'll Learn

1. **Data Generation**: Create synthetic medical imaging data that mimics real MRI characteristics
2. **Data Preprocessing**: Normalize, augment, and prepare images for training
3. **Model Architecture**: Build a Convolutional Neural Network (CNN) from scratch
4. **Training Pipeline**: Implement training loops with validation and checkpointing
5. **Evaluation Metrics**: Assess model performance using clinical-relevant metrics
6. **Interactive Dashboard**: Visualize results with comprehensive dashboards

### Prerequisites

- Basic understanding of Python and machine learning
- Familiarity with NumPy and PyTorch (helpful but not required)
- Understanding of binary classification concepts

### Medical Context

In clinical practice, distinguishing between benign and malignant tumors is crucial:
- **Benign tumors**: Non-cancerous, typically don't spread, usually have clear boundaries
- **Malignant tumors**: Cancerous, can metastasize, often have irregular shapes

Deep learning models can assist radiologists by:
- Providing second opinions
- Reducing analysis time
- Detecting subtle patterns humans might miss
- Standardizing diagnostic criteria

**Note**: This is an educational tutorial using synthetic data. Real medical AI systems require extensive validation, regulatory approval, and should never replace professional medical judgment.

## 1. Setup and Imports

First, let's import all necessary libraries and set up our environment.

In [ ]:
# Core libraries
import os
import numpy as np
import pandas as pd
from pathlib import Path
import random
from typing import Tuple, List, Dict
import warnings
warnings.filterwarnings('ignore')

# Deep learning framework
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms

# Image processing
from PIL import Image

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Metrics and evaluation
from sklearn.metrics import (
    confusion_matrix, 
    classification_report,
    roc_curve, 
    auc,
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Progress bar
from tqdm.auto import tqdm

# Set random seeds for reproducibility
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n✓ All imports successful!")
print(f"PyTorch version: {torch.__version__}")

## 2. Synthetic Medical Image Data Generation

Since we're working with a tutorial, we'll create synthetic MRI-like images that simulate tumor characteristics.

### Image Characteristics

**Benign tumors**:
- Round or oval shape
- Smooth, well-defined boundaries
- Uniform texture
- Lower intensity variation

**Malignant tumors**:
- Irregular, asymmetric shape
- Poorly defined edges
- Heterogeneous texture
- Higher intensity variation
- May have satellite lesions

In [ ]:
class SyntheticMRIGenerator:
    """
    Generates synthetic MRI-like images for tumor classification.
    
    This class creates grayscale images that simulate the appearance of MRI scans
    with benign or malignant tumors.
    """
    
    def __init__(self, img_size: int = 128, noise_level: float = 0.1):
        """
        Args:
            img_size: Size of the generated square images (default 128x128)
            noise_level: Amount of random noise to add (0-1 scale)
        """
        self.img_size = img_size
        self.noise_level = noise_level
        
    def generate_benign_tumor(self) -> np.ndarray:
        """
        Generate a synthetic benign tumor image.
        
        Benign tumors are characterized by:
        - Regular circular/oval shape
        - Smooth boundaries
        - Homogeneous internal structure
        """
        img = np.zeros((self.img_size, self.img_size), dtype=np.float32)
        
        # Random center position (avoid edges)
        center_x = np.random.randint(self.img_size // 4, 3 * self.img_size // 4)
        center_y = np.random.randint(self.img_size // 4, 3 * self.img_size // 4)
        
        # Regular size (relatively uniform)
        radius = np.random.randint(self.img_size // 8, self.img_size // 5)
        
        # Create smooth circular tumor
        y, x = np.ogrid[:self.img_size, :self.img_size]
        mask = (x - center_x)**2 + (y - center_y)**2 <= radius**2
        
        # Uniform intensity with slight gradient
        base_intensity = np.random.uniform(0.6, 0.9)
        img[mask] = base_intensity
        
        # Add subtle texture
        img += np.random.normal(0, 0.05, (self.img_size, self.img_size))
        
        # Add background tissue appearance
        background = np.random.normal(0.2, 0.05, (self.img_size, self.img_size))
        img = np.where(mask, img, background)
        
        # Gaussian blur for smooth edges
        from scipy.ndimage import gaussian_filter
        img = gaussian_filter(img, sigma=2.0)
        
        return np.clip(img, 0, 1)
    
    def generate_malignant_tumor(self) -> np.ndarray:
        """
        Generate a synthetic malignant tumor image.
        
        Malignant tumors are characterized by:
        - Irregular, asymmetric shape
        - Spiky or poorly defined boundaries
        - Heterogeneous texture
        - Possible satellite lesions
        """
        img = np.zeros((self.img_size, self.img_size), dtype=np.float32)
        
        # Random center
        center_x = np.random.randint(self.img_size // 4, 3 * self.img_size // 4)
        center_y = np.random.randint(self.img_size // 4, 3 * self.img_size // 4)
        
        # Irregular size
        base_radius = np.random.randint(self.img_size // 8, self.img_size // 4)
        
        # Create irregular shape using multiple overlapping circles
        y, x = np.ogrid[:self.img_size, :self.img_size]
        
        # Main tumor mass (irregular)
        n_lobes = np.random.randint(3, 6)
        for i in range(n_lobes):
            angle = 2 * np.pi * i / n_lobes + np.random.uniform(-0.3, 0.3)
            offset_x = int(base_radius * 0.5 * np.cos(angle))
            offset_y = int(base_radius * 0.5 * np.sin(angle))
            lobe_radius = base_radius * np.random.uniform(0.6, 1.0)
            
            mask = ((x - center_x - offset_x)**2 + 
                    (y - center_y - offset_y)**2 <= lobe_radius**2)
            
            # Heterogeneous intensity
            intensity = np.random.uniform(0.5, 0.95)
            img[mask] = np.maximum(img[mask], intensity)
        
        # Add satellite lesions (characteristic of malignancy)
        n_satellites = np.random.randint(1, 4)
        for _ in range(n_satellites):
            sat_x = center_x + np.random.randint(-base_radius, base_radius)
            sat_y = center_y + np.random.randint(-base_radius, base_radius)
            sat_radius = np.random.randint(3, 8)
            
            mask = ((x - sat_x)**2 + (y - sat_y)**2 <= sat_radius**2)
            img[mask] = np.random.uniform(0.6, 0.9)
        
        # Add heterogeneous texture (necrosis, hemorrhage)
        img += np.random.normal(0, 0.15, (self.img_size, self.img_size))
        
        # Add background
        background = np.random.normal(0.2, 0.05, (self.img_size, self.img_size))
        img = np.where(img > 0.3, img, background)
        
        # Less smoothing for irregular edges
        from scipy.ndimage import gaussian_filter
        img = gaussian_filter(img, sigma=1.0)
        
        return np.clip(img, 0, 1)
    
    def generate_dataset(self, n_samples: int, save_dir: Path) -> pd.DataFrame:
        """
        Generate a balanced dataset of benign and malignant tumor images.
        
        Args:
            n_samples: Total number of images to generate
            save_dir: Directory to save images
            
        Returns:
            DataFrame with image paths and labels
        """
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        
        # Create subdirectories
        (save_dir / 'benign').mkdir(exist_ok=True)
        (save_dir / 'malignant').mkdir(exist_ok=True)
        
        data = []
        
        print(f"Generating {n_samples} synthetic MRI images...")
        
        for i in tqdm(range(n_samples)):
            # Alternate between benign and malignant for balance
            if i % 2 == 0:
                img = self.generate_benign_tumor()
                label = 0  # Benign
                subdir = 'benign'
            else:
                img = self.generate_malignant_tumor()
                label = 1  # Malignant
                subdir = 'malignant'
            
            # Save image
            filename = f"{subdir}_{i:05d}.png"
            filepath = save_dir / subdir / filename
            
            # Convert to 8-bit grayscale and save
            img_8bit = (img * 255).astype(np.uint8)
            Image.fromarray(img_8bit, mode='L').save(filepath)
            
            data.append({
                'image_path': str(filepath),
                'label': label,
                'class_name': 'malignant' if label == 1 else 'benign'
            })
        
        df = pd.DataFrame(data)
        
        print(f"\n✓ Generated {len(df)} images")
        print(f"  - Benign: {(df['label'] == 0).sum()}")
        print(f"  - Malignant: {(df['label'] == 1).sum()}")
        
        return df

print("✓ SyntheticMRIGenerator class defined")

### Generate the Dataset

Let's create our synthetic dataset. We'll generate 2000 images (1000 benign, 1000 malignant).

In [ ]:
# Install scipy if not available
try:
    import scipy
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scipy"])
    import scipy

# Setup paths
DATA_DIR = Path('../data/synthetic_mri')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Generate dataset
generator = SyntheticMRIGenerator(img_size=128, noise_level=0.1)
df_data = generator.generate_dataset(n_samples=2000, save_dir=DATA_DIR)

# Save metadata
metadata_path = DATA_DIR / 'metadata.csv'
df_data.to_csv(metadata_path, index=False)
print(f"\n✓ Metadata saved to {metadata_path}")

### Visualize Sample Images

Let's look at some examples to understand the differences between benign and malignant tumors.

In [ ]:
def visualize_samples(df: pd.DataFrame, n_samples: int = 8):
    """
    Visualize sample images from each class.
    """
    fig, axes = plt.subplots(2, n_samples // 2, figsize=(20, 8))
    
    # Get samples
    benign_samples = df[df['label'] == 0].sample(n_samples // 2, random_state=RANDOM_SEED)
    malignant_samples = df[df['label'] == 1].sample(n_samples // 2, random_state=RANDOM_SEED)
    
    # Plot benign
    for i, (idx, row) in enumerate(benign_samples.iterrows()):
        img = Image.open(row['image_path'])
        axes[0, i].imshow(img, cmap='gray')
        axes[0, i].set_title('BENIGN', fontsize=12, fontweight='bold', color='green')
        axes[0, i].axis('off')
    
    # Plot malignant
    for i, (idx, row) in enumerate(malignant_samples.iterrows()):
        img = Image.open(row['image_path'])
        axes[1, i].imshow(img, cmap='gray')
        axes[1, i].set_title('MALIGNANT', fontsize=12, fontweight='bold', color='red')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.suptitle('Sample Tumor MRI Images', fontsize=16, fontweight='bold', y=1.02)
    plt.show()
    
    print("\nKey Differences to Notice:")
    print("  Benign (Top Row):")
    print("    • Regular, round/oval shape")
    print("    • Smooth, well-defined boundaries")
    print("    • Uniform internal texture")
    print("\n  Malignant (Bottom Row):")
    print("    • Irregular, asymmetric shape")
    print("    • Spiky or poorly defined edges")
    print("    • Heterogeneous texture with variations")
    print("    • May have small satellite lesions")

visualize_samples(df_data)

## 3. Data Loading and Preprocessing

Now we'll create a PyTorch Dataset and DataLoader to efficiently load and preprocess our images during training.

### Data Augmentation

Data augmentation helps prevent overfitting by creating variations of training images:
- **Random rotations**: Simulates different scan orientations
- **Random flips**: Accounts for anatomical variations
- **Brightness/contrast adjustments**: Simulates different scanner settings
- **Random crops**: Helps model focus on tumor features

In [ ]:
class TumorDataset(Dataset):
    """
    PyTorch Dataset for tumor MRI images.
    """
    
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        """
        Args:
            dataframe: DataFrame with 'image_path' and 'label' columns
            transform: Optional transform to apply to images
        """
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        # Load image
        img_path = self.dataframe.loc[idx, 'image_path']
        image = Image.open(img_path).convert('L')  # Grayscale
        
        # Get label
        label = self.dataframe.loc[idx, 'label']
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        return image, label


# Define transforms
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # Normalize to [-1, 1]
])

val_test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

print("✓ TumorDataset class and transforms defined")

### Split Data into Train/Val/Test Sets

We'll use a 70/15/15 split:
- **Training set (70%)**: Used to train the model
- **Validation set (15%)**: Used to tune hyperparameters and monitor training
- **Test set (15%)**: Final evaluation, never seen during training

In [ ]:
# Split data
from sklearn.model_selection import train_test_split

# First split: train + val vs test
train_val_df, test_df = train_test_split(
    df_data, 
    test_size=0.15, 
    stratify=df_data['label'],
    random_state=RANDOM_SEED
)

# Second split: train vs val
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.15 / 0.85,  # 15% of total
    stratify=train_val_df['label'],
    random_state=RANDOM_SEED
)

print("Dataset Split:")
print(f"  Training:   {len(train_df)} images ({len(train_df) / len(df_data) * 100:.1f}%)")
print(f"  Validation: {len(val_df)} images ({len(val_df) / len(df_data) * 100:.1f}%)")
print(f"  Test:       {len(test_df)} images ({len(test_df) / len(df_data) * 100:.1f}%)")

# Create datasets
train_dataset = TumorDataset(train_df, transform=train_transform)
val_dataset = TumorDataset(val_df, transform=val_test_transform)
test_dataset = TumorDataset(test_df, transform=val_test_transform)

# Create dataloaders
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True if device.type == 'cuda' else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if device.type == 'cuda' else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if device.type == 'cuda' else False
)

print(f"\n✓ DataLoaders created with batch size {BATCH_SIZE}")
print(f"  Training batches:   {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")
print(f"  Test batches:       {len(test_loader)}")

### Visualize Data Augmentation

Let's see how data augmentation creates variations of the same image.

In [ ]:
def visualize_augmentation(dataset: Dataset, idx: int = 0, n_augmentations: int = 8):
    """
    Show the same image with different random augmentations.
    """
    fig, axes = plt.subplots(2, n_augmentations // 2, figsize=(20, 8))
    axes = axes.flatten()
    
    for i in range(n_augmentations):
        img, label = dataset[idx]
        
        # Convert to numpy for plotting
        img_np = img.squeeze().numpy()
        
        axes[i].imshow(img_np, cmap='gray')
        axes[i].set_title(f'Augmentation {i+1}', fontsize=10)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.suptitle(f'Data Augmentation Examples - Class: {"Malignant" if label == 1 else "Benign"}',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.show()

# Show augmentations for one image
visualize_augmentation(train_dataset, idx=0)

## 4. CNN Model Architecture

Now let's build our Convolutional Neural Network for tumor classification.

### Architecture Design

Our CNN consists of:
1. **Convolutional Blocks**: Extract spatial features at different scales
   - Conv2D → BatchNorm → ReLU → MaxPool
2. **Global Average Pooling**: Reduces spatial dimensions while preserving features
3. **Fully Connected Layers**: Final classification with dropout for regularization

### Why This Architecture?

- **Convolutional layers**: Detect edges, textures, and shapes (tumor boundaries, texture patterns)
- **Batch normalization**: Stabilizes training and improves convergence
- **Dropout**: Prevents overfitting by randomly dropping neurons during training
- **Global average pooling**: More robust than flattening, reduces overfitting

In [ ]:
class TumorClassifierCNN(nn.Module):
    """
    Convolutional Neural Network for binary tumor classification.
    
    Architecture:
        - 4 convolutional blocks with increasing depth
        - Batch normalization and dropout for regularization
        - Global average pooling
        - Fully connected layers for classification
    """
    
    def __init__(self, dropout_rate: float = 0.5):
        super(TumorClassifierCNN, self).__init__()
        
        # Convolutional Block 1
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=0.1)
        )
        
        # Convolutional Block 2
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=0.2)
        )
        
        # Convolutional Block 3
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=0.3)
        )
        
        # Convolutional Block 4
        self.conv_block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=0.4)
        )
        
        # Global Average Pooling
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Fully Connected Layers
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),
            nn.Linear(64, 1)  # Binary classification
        )
    
    def forward(self, x):
        """
        Forward pass through the network.
        
        Args:
            x: Input tensor of shape (batch_size, 1, 128, 128)
            
        Returns:
            Output logits of shape (batch_size, 1)
        """
        x = self.conv_block1(x)  # (B, 32, 64, 64)
        x = self.conv_block2(x)  # (B, 64, 32, 32)
        x = self.conv_block3(x)  # (B, 128, 16, 16)
        x = self.conv_block4(x)  # (B, 256, 8, 8)
        
        x = self.global_avg_pool(x)  # (B, 256, 1, 1)
        x = self.classifier(x)  # (B, 1)
        
        return x


# Initialize model
model = TumorClassifierCNN(dropout_rate=0.5).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✓ Model initialized")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

## 5. Training Pipeline

Now let's implement the training loop with validation monitoring.

### Training Strategy

- **Loss function**: Binary Cross Entropy with Logits (BCEWithLogitsLoss)
  - Combines sigmoid activation and BCE loss for numerical stability
- **Optimizer**: Adam with weight decay (L2 regularization)
- **Learning rate scheduler**: Reduce on plateau to fine-tune training
- **Early stopping**: Stop if validation loss doesn't improve
- **Model checkpointing**: Save best model based on validation performance

In [ ]:
class ModelTrainer:
    """
    Handles training, validation, and checkpointing for the tumor classifier.
    """
    
    def __init__(
        self,
        model: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        device: torch.device,
        learning_rate: float = 1e-3,
        weight_decay: float = 1e-4
    ):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        
        # Loss function
        self.criterion = nn.BCEWithLogitsLoss()
        
        # Optimizer
        self.optimizer = optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay
        )
        
        # Learning rate scheduler
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode='min',
            factor=0.5,
            patience=5,
            verbose=True
        )
        
        # Training history
        self.history = {
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': [],
            'learning_rates': []
        }
        
        self.best_val_loss = float('inf')
        self.epochs_no_improve = 0
    
    def train_epoch(self) -> Tuple[float, float]:
        """
        Train for one epoch.
        
        Returns:
            Tuple of (average_loss, accuracy)
        """
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(self.train_loader, desc='Training', leave=False)
        for images, labels in pbar:
            images = images.to(self.device)
            labels = labels.float().unsqueeze(1).to(self.device)
            
            # Forward pass
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            
            # Backward pass
            loss.backward()
            self.optimizer.step()
            
            # Statistics
            running_loss += loss.item() * images.size(0)
            predictions = (torch.sigmoid(outputs) > 0.5).float()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
            
            # Update progress bar
            pbar.set_postfix({
                'loss': loss.item(),
                'acc': 100. * correct / total
            })
        
        epoch_loss = running_loss / total
        epoch_acc = 100. * correct / total
        
        return epoch_loss, epoch_acc
    
    def validate(self) -> Tuple[float, float]:
        """
        Validate the model.
        
        Returns:
            Tuple of (average_loss, accuracy)
        """
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in self.val_loader:
                images = images.to(self.device)
                labels = labels.float().unsqueeze(1).to(self.device)
                
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                
                running_loss += loss.item() * images.size(0)
                predictions = (torch.sigmoid(outputs) > 0.5).float()
                correct += (predictions == labels).sum().item()
                total += labels.size(0)
        
        epoch_loss = running_loss / total
        epoch_acc = 100. * correct / total
        
        return epoch_loss, epoch_acc
    
    def train(
        self,
        n_epochs: int,
        save_dir: Path,
        early_stopping_patience: int = 10
    ) -> Dict:
        """
        Train the model for multiple epochs.
        
        Args:
            n_epochs: Number of epochs to train
            save_dir: Directory to save model checkpoints
            early_stopping_patience: Stop if no improvement for this many epochs
            
        Returns:
            Training history dictionary
        """
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"Starting training for {n_epochs} epochs...\n")
        
        for epoch in range(1, n_epochs + 1):
            print(f"Epoch {epoch}/{n_epochs}")
            print("-" * 50)
            
            # Train
            train_loss, train_acc = self.train_epoch()
            
            # Validate
            val_loss, val_acc = self.validate()
            
            # Update scheduler
            self.scheduler.step(val_loss)
            current_lr = self.optimizer.param_groups[0]['lr']
            
            # Save history
            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)
            self.history['learning_rates'].append(current_lr)
            
            # Print epoch results
            print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
            print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
            print(f"Learning Rate: {current_lr:.2e}")
            
            # Save best model
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.epochs_no_improve = 0
                
                checkpoint_path = save_dir / 'best_model.pth'
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_loss': val_loss,
                    'val_acc': val_acc,
                }, checkpoint_path)
                
                print(f"✓ Best model saved (val_loss: {val_loss:.4f})")
            else:
                self.epochs_no_improve += 1
            
            # Early stopping
            if self.epochs_no_improve >= early_stopping_patience:
                print(f"\nEarly stopping triggered after {epoch} epochs")
                print(f"No improvement in validation loss for {early_stopping_patience} epochs")
                break
            
            print()
        
        print("\n" + "="*50)
        print("Training completed!")
        print(f"Best validation loss: {self.best_val_loss:.4f}")
        print("="*50)
        
        return self.history

print("✓ ModelTrainer class defined")

### Start Training

Let's train our model! This will take a few minutes depending on your hardware.

In [ ]:
# Setup training
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

trainer = ModelTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    learning_rate=1e-3,
    weight_decay=1e-4
)

# Train model
history = trainer.train(
    n_epochs=50,
    save_dir=MODELS_DIR,
    early_stopping_patience=10
)

### Visualize Training History

Let's examine how the model learned over time.

In [ ]:
def plot_training_history(history: Dict):
    """
    Plot training and validation metrics over epochs.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss
    axes[0].plot(epochs, history['train_loss'], 'b-', label='Training Loss', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(epochs, history['train_acc'], 'b-', label='Training Accuracy', linewidth=2)
    axes[1].plot(epochs, history['val_acc'], 'r-', label='Validation Accuracy', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    
    # Learning Rate
    axes[2].plot(epochs, history['learning_rates'], 'g-', linewidth=2)
    axes[2].set_xlabel('Epoch', fontsize=12)
    axes[2].set_ylabel('Learning Rate', fontsize=12)
    axes[2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    axes[2].set_yscale('log')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print final metrics
    print("\nFinal Training Metrics:")
    print(f"  Training Loss:   {history['train_loss'][-1]:.4f}")
    print(f"  Training Acc:    {history['train_acc'][-1]:.2f}%")
    print(f"  Validation Loss: {history['val_loss'][-1]:.4f}")
    print(f"  Validation Acc:  {history['val_acc'][-1]:.2f}%")

plot_training_history(history)

## 6. Model Evaluation

Now let's thoroughly evaluate our trained model on the test set.

### Clinical Metrics

For medical AI, we care about:
- **Sensitivity (Recall)**: How many actual malignant tumors did we catch?
- **Specificity**: How many benign tumors did we correctly identify?
- **Precision**: Of our malignant predictions, how many were correct?
- **F1-Score**: Harmonic mean of precision and recall
- **ROC-AUC**: Overall discrimination ability

In [ ]:
def evaluate_model(model: nn.Module, test_loader: DataLoader, device: torch.device) -> Dict:
    """
    Comprehensive model evaluation.
    
    Returns:
        Dictionary with predictions, labels, and probabilities
    """
    model.eval()
    
    all_predictions = []
    all_labels = []
    all_probabilities = []
    
    print("Evaluating model on test set...")
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc='Testing'):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            probabilities = torch.sigmoid(outputs).squeeze()
            predictions = (probabilities > 0.5).float()
            
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    
    results = {
        'predictions': np.array(all_predictions),
        'labels': np.array(all_labels),
        'probabilities': np.array(all_probabilities)
    }
    
    return results


# Load best model
checkpoint = torch.load(MODELS_DIR / 'best_model.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✓ Loaded best model (epoch {checkpoint['epoch']}, val_loss: {checkpoint['val_loss']:.4f})\n")

# Evaluate
results = evaluate_model(model, test_loader, device)

### Calculate Metrics

In [ ]:
# Calculate metrics
y_true = results['labels']
y_pred = results['predictions']
y_prob = results['probabilities']

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)  # Sensitivity
f1 = f1_score(y_true, y_pred)

# Calculate specificity
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
specificity = tn / (tn + fp)

# ROC-AUC
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

# Print metrics
print("\n" + "="*60)
print("TEST SET EVALUATION METRICS")
print("="*60)
print(f"\nOverall Performance:")
print(f"  Accuracy:   {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  ROC-AUC:    {roc_auc:.4f}")
print(f"  F1-Score:   {f1:.4f}")
print(f"\nClinical Metrics:")
print(f"  Sensitivity (Recall):  {recall:.4f} - Detected {recall*100:.1f}% of malignant tumors")
print(f"  Specificity:           {specificity:.4f} - Correctly identified {specificity*100:.1f}% of benign tumors")
print(f"  Precision:             {precision:.4f} - {precision*100:.1f}% of malignant predictions were correct")
print(f"\nConfusion Matrix Breakdown:")
print(f"  True Positives  (Correctly identified malignant): {tp}")
print(f"  True Negatives  (Correctly identified benign):    {tn}")
print(f"  False Positives (Benign misclassified as malignant): {fp}")
print(f"  False Negatives (Malignant misclassified as benign): {fn}")
print("="*60)

# Store metrics for dashboard
metrics_dict = {
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'specificity': specificity,
    'f1_score': f1,
    'roc_auc': roc_auc,
    'confusion_matrix': {'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn}
}

## 7. Interactive Classification Dashboard

Now let's create comprehensive, interactive visualizations to understand our model's performance.

The dashboard includes:
1. **Confusion Matrix**: Visual breakdown of predictions
2. **ROC Curve**: Trade-off between sensitivity and specificity
3. **Precision-Recall Curve**: Performance at different thresholds
4. **Metrics Summary**: Key performance indicators
5. **Prediction Distribution**: How confident is the model?
6. **Sample Predictions**: Visual examples of correct and incorrect classifications

In [ ]:
def create_interactive_dashboard(results: Dict, metrics: Dict):
    """
    Create an interactive dashboard using Plotly for model evaluation.
    """
    y_true = results['labels']
    y_pred = results['predictions']
    y_prob = results['probabilities']
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=(
            'Confusion Matrix',
            'ROC Curve',
            'Precision-Recall Curve',
            'Metrics Summary',
            'Prediction Probability Distribution',
            'Calibration: Prediction vs True Class'
        ),
        specs=[
            [{'type': 'heatmap'}, {'type': 'scatter'}, {'type': 'scatter'}],
            [{'type': 'bar'}, {'type': 'histogram'}, {'type': 'box'}]
        ],
        vertical_spacing=0.12,
        horizontal_spacing=0.10
    )
    
    # 1. Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    fig.add_trace(
        go.Heatmap(
            z=cm_normalized,
            x=['Benign', 'Malignant'],
            y=['Benign', 'Malignant'],
            text=cm,
            texttemplate='%{text}',
            colorscale='Blues',
            showscale=True,
            hovertemplate='True: %{y}<br>Predicted: %{x}<br>Count: %{text}<br>Rate: %{z:.2%}<extra></extra>'
        ),
        row=1, col=1
    )
    
    # 2. ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    
    fig.add_trace(
        go.Scatter(
            x=fpr,
            y=tpr,
            mode='lines',
            name=f'ROC (AUC = {roc_auc:.3f})',
            line=dict(color='darkorange', width=2),
            hovertemplate='FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>'
        ),
        row=1, col=2
    )
    
    # Add diagonal reference line
    fig.add_trace(
        go.Scatter(
            x=[0, 1],
            y=[0, 1],
            mode='lines',
            name='Random Classifier',
            line=dict(color='navy', width=2, dash='dash'),
            showlegend=False
        ),
        row=1, col=2
    )
    
    # 3. Precision-Recall Curve
    precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall_vals, precision_vals)
    
    fig.add_trace(
        go.Scatter(
            x=recall_vals,
            y=precision_vals,
            mode='lines',
            name=f'PR (AUC = {pr_auc:.3f})',
            line=dict(color='green', width=2),
            hovertemplate='Recall: %{x:.3f}<br>Precision: %{y:.3f}<extra></extra>'
        ),
        row=1, col=3
    )
    
    # 4. Metrics Bar Chart
    metric_names = ['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1-Score', 'ROC-AUC']
    metric_values = [
        metrics['accuracy'],
        metrics['precision'],
        metrics['recall'],
        metrics['specificity'],
        metrics['f1_score'],
        metrics['roc_auc']
    ]
    
    colors = ['lightblue' if v < 0.8 else 'lightgreen' if v < 0.9 else 'green' 
              for v in metric_values]
    
    fig.add_trace(
        go.Bar(
            x=metric_names,
            y=metric_values,
            text=[f'{v:.3f}' for v in metric_values],
            textposition='outside',
            marker_color=colors,
            hovertemplate='%{x}: %{y:.4f}<extra></extra>'
        ),
        row=2, col=1
    )
    
    # 5. Prediction Probability Distribution
    benign_probs = y_prob[y_true == 0]
    malignant_probs = y_prob[y_true == 1]
    
    fig.add_trace(
        go.Histogram(
            x=benign_probs,
            name='Benign (True)',
            opacity=0.7,
            marker_color='lightblue',
            nbinsx=30,
            hovertemplate='Probability: %{x:.2f}<br>Count: %{y}<extra></extra>'
        ),
        row=2, col=2
    )
    
    fig.add_trace(
        go.Histogram(
            x=malignant_probs,
            name='Malignant (True)',
            opacity=0.7,
            marker_color='salmon',
            nbinsx=30,
            hovertemplate='Probability: %{x:.2f}<br>Count: %{y}<extra></extra>'
        ),
        row=2, col=2
    )
    
    # 6. Box Plot of Probabilities by True Class
    fig.add_trace(
        go.Box(
            y=benign_probs,
            name='Benign',
            marker_color='lightblue',
            boxmean='sd'
        ),
        row=2, col=3
    )
    
    fig.add_trace(
        go.Box(
            y=malignant_probs,
            name='Malignant',
            marker_color='salmon',
            boxmean='sd'
        ),
        row=2, col=3
    )
    
    # Update layout
    fig.update_xaxes(title_text="Predicted", row=1, col=1)
    fig.update_yaxes(title_text="True", row=1, col=1)
    
    fig.update_xaxes(title_text="False Positive Rate", row=1, col=2)
    fig.update_yaxes(title_text="True Positive Rate", row=1, col=2)
    
    fig.update_xaxes(title_text="Recall", row=1, col=3)
    fig.update_yaxes(title_text="Precision", row=1, col=3)
    
    fig.update_yaxes(title_text="Score", range=[0, 1.1], row=2, col=1)
    
    fig.update_xaxes(title_text="Predicted Probability (Malignant)", row=2, col=2)
    fig.update_yaxes(title_text="Count", row=2, col=2)
    
    fig.update_yaxes(title_text="Predicted Probability", row=2, col=3)
    
    # Update overall layout
    fig.update_layout(
        title_text="<b>Tumor Classification Model - Comprehensive Evaluation Dashboard</b>",
        title_x=0.5,
        title_font_size=20,
        showlegend=True,
        height=900,
        hovermode='closest',
        template='plotly_white'
    )
    
    fig.show()
    
    print("\n✓ Interactive dashboard created successfully!")
    print("\nDashboard Components:")
    print("  1. Confusion Matrix: Shows the breakdown of correct and incorrect predictions")
    print("  2. ROC Curve: Illustrates the trade-off between true positive and false positive rates")
    print("  3. Precision-Recall Curve: Shows precision vs recall trade-off")
    print("  4. Metrics Summary: Bar chart of all key performance metrics")
    print("  5. Probability Distribution: How confident the model is in its predictions")
    print("  6. Calibration Plot: Box plots showing prediction distribution by true class")

# Create the dashboard
create_interactive_dashboard(results, metrics_dict)

### Visualize Sample Predictions

Let's look at some specific examples to understand where the model succeeds and fails.

In [ ]:
def visualize_predictions(test_dataset: Dataset, results: Dict, n_samples: int = 8):
    """
    Visualize sample predictions with confidence scores.
    """
    y_true = results['labels']
    y_pred = results['predictions']
    y_prob = results['probabilities']
    
    # Get indices for different categories
    correct_indices = np.where(y_pred == y_true)[0]
    incorrect_indices = np.where(y_pred != y_true)[0]
    
    # Sample from each category
    if len(correct_indices) >= n_samples // 2:
        correct_sample = np.random.choice(correct_indices, n_samples // 2, replace=False)
    else:
        correct_sample = correct_indices
    
    if len(incorrect_indices) >= n_samples // 2:
        incorrect_sample = np.random.choice(incorrect_indices, n_samples // 2, replace=False)
    else:
        incorrect_sample = incorrect_indices
    
    # Combine samples
    sample_indices = np.concatenate([correct_sample, incorrect_sample])
    
    fig, axes = plt.subplots(2, len(sample_indices) // 2, figsize=(20, 8))
    axes = axes.flatten()
    
    for i, idx in enumerate(sample_indices):
        # Get image
        img, true_label = test_dataset[idx]
        img_np = img.squeeze().numpy()
        
        pred_label = y_pred[idx]
        prob = y_prob[idx]
        
        # Determine if prediction is correct
        is_correct = pred_label == true_label
        
        # Class names
        true_class = 'Malignant' if true_label == 1 else 'Benign'
        pred_class = 'Malignant' if pred_label == 1 else 'Benign'
        
        # Plot
        axes[i].imshow(img_np, cmap='gray')
        
        # Color code by correctness
        color = 'green' if is_correct else 'red'
        status = '✓ CORRECT' if is_correct else '✗ INCORRECT'
        
        title = f"{status}\nTrue: {true_class}\nPred: {pred_class} ({prob:.2%})"
        axes[i].set_title(title, fontsize=10, fontweight='bold', color=color)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.suptitle('Sample Predictions with Confidence Scores', 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.show()
    
    print("\nInterpretation Guide:")
    print("  • Green titles: Correct predictions")
    print("  • Red titles: Incorrect predictions")
    print("  • Percentage shown: Model's confidence in malignant prediction")
    print("    - Close to 0%: High confidence it's benign")
    print("    - Close to 100%: High confidence it's malignant")
    print("    - Around 50%: Model is uncertain")

visualize_predictions(test_dataset, results, n_samples=8)

## 8. Clinical Insights and Model Interpretation

Let's analyze our model's performance from a clinical perspective.

In [ ]:
def generate_clinical_report(metrics: Dict, results: Dict):
    """
    Generate a clinical-style report of model performance.
    """
    cm = metrics['confusion_matrix']
    
    print("\n" + "="*70)
    print("CLINICAL PERFORMANCE REPORT")
    print("Tumor Classification Model - Benign vs Malignant")
    print("="*70)
    
    print("\n1. OVERALL DIAGNOSTIC ACCURACY")
    print("-" * 70)
    print(f"   Model correctly classified {metrics['accuracy']*100:.1f}% of cases")
    print(f"   Overall discriminative ability (ROC-AUC): {metrics['roc_auc']:.3f}")
    
    print("\n2. MALIGNANT TUMOR DETECTION (Most Critical)")
    print("-" * 70)
    print(f"   Sensitivity (True Positive Rate): {metrics['recall']*100:.1f}%")
    print(f"     → Detected {cm['TP']} out of {cm['TP'] + cm['FN']} malignant cases")
    print(f"     → Missed {cm['FN']} malignant cases (False Negatives)")
    
    if metrics['recall'] >= 0.95:
        print("     ✓ EXCELLENT: Very few malignant cases missed")
    elif metrics['recall'] >= 0.85:
        print("     ⚠ GOOD: Acceptable detection rate, but room for improvement")
    else:
        print("     ✗ CONCERN: Missing too many malignant cases for clinical use")
    
    print("\n3. BENIGN TUMOR IDENTIFICATION")
    print("-" * 70)
    print(f"   Specificity (True Negative Rate): {metrics['specificity']*100:.1f}%")
    print(f"     → Correctly identified {cm['TN']} out of {cm['TN'] + cm['FP']} benign cases")
    print(f"     → Misclassified {cm['FP']} benign as malignant (False Positives)")
    
    if metrics['specificity'] >= 0.90:
        print("     ✓ EXCELLENT: Minimizes unnecessary anxiety and procedures")
    elif metrics['specificity'] >= 0.80:
        print("     ⚠ ACCEPTABLE: Some false alarms, but manageable")
    else:
        print("     ✗ CONCERN: Too many false positives may cause unnecessary stress")
    
    print("\n4. POSITIVE PREDICTIVE VALUE")
    print("-" * 70)
    print(f"   Precision: {metrics['precision']*100:.1f}%")
    print(f"     → When model predicts malignant, it's correct {metrics['precision']*100:.1f}% of the time")
    print(f"     → {cm['FP']} patients might undergo unnecessary follow-up procedures")
    
    print("\n5. CLINICAL RISK ASSESSMENT")
    print("-" * 70)
    
    # False Negative Risk (missed malignancies)
    fn_rate = cm['FN'] / (cm['TP'] + cm['FN'])
    print(f"   ⚠ False Negative Rate: {fn_rate*100:.1f}%")
    print(f"     Risk: {cm['FN']} malignant tumors might go untreated")
    print(f"     Impact: HIGH - Potential delayed diagnosis and treatment")
    
    # False Positive Risk
    fp_rate = cm['FP'] / (cm['TN'] + cm['FP'])
    print(f"\n   ⚠ False Positive Rate: {fp_rate*100:.1f}%")
    print(f"     Risk: {cm['FP']} patients might undergo unnecessary procedures")
    print(f"     Impact: MODERATE - Psychological stress and unnecessary testing")
    
    print("\n6. RECOMMENDATIONS FOR CLINICAL DEPLOYMENT")
    print("-" * 70)
    
    recommendations = []
    
    if metrics['recall'] < 0.95:
        recommendations.append(
            "   • Increase sensitivity through threshold adjustment or model retraining"
        )
    
    if metrics['specificity'] < 0.85:
        recommendations.append(
            "   • Reduce false positives to minimize patient anxiety and unnecessary procedures"
        )
    
    recommendations.extend([
        "   • Use as a DECISION SUPPORT tool, not autonomous diagnosis",
        "   • Always require radiologist confirmation of predictions",
        "   • Implement confidence thresholds - flag uncertain cases for expert review",
        "   • Regular model retraining with new diverse data",
        "   • Monitor performance across different patient demographics",
        "   • Validate on real clinical data before deployment"
    ])
    
    for rec in recommendations:
        print(rec)
    
    print("\n" + "="*70)
    print("END OF CLINICAL REPORT")
    print("="*70)

generate_clinical_report(metrics_dict, results)

## 9. Save Model and Results

Let's save everything for future use.

In [ ]:
import json

# Create outputs directory
OUTPUTS_DIR = Path('../outputs')
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Save training history
history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUTS_DIR / 'training_history.csv', index=False)
print(f"✓ Training history saved to {OUTPUTS_DIR / 'training_history.csv'}")

# Save evaluation metrics
with open(OUTPUTS_DIR / 'evaluation_metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)
print(f"✓ Evaluation metrics saved to {OUTPUTS_DIR / 'evaluation_metrics.json'}")

# Save predictions
predictions_df = pd.DataFrame({
    'true_label': results['labels'],
    'predicted_label': results['predictions'],
    'probability_malignant': results['probabilities']
})
predictions_df.to_csv(OUTPUTS_DIR / 'test_predictions.csv', index=False)
print(f"✓ Test predictions saved to {OUTPUTS_DIR / 'test_predictions.csv'}")

# Save model summary
summary = {
    'model_architecture': 'TumorClassifierCNN',
    'total_parameters': total_params,
    'input_size': '128x128 grayscale',
    'output': 'Binary classification (Benign vs Malignant)',
    'training_samples': len(train_df),
    'validation_samples': len(val_df),
    'test_samples': len(test_df),
    'best_epoch': checkpoint['epoch'],
    'best_val_loss': float(checkpoint['val_loss']),
    'test_accuracy': float(metrics_dict['accuracy']),
    'test_roc_auc': float(metrics_dict['roc_auc'])
}

with open(OUTPUTS_DIR / 'model_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✓ Model summary saved to {OUTPUTS_DIR / 'model_summary.json'}")

print("\n" + "="*70)
print("ALL RESULTS SAVED SUCCESSFULLY!")
print("="*70)
print(f"\nOutput files location: {OUTPUTS_DIR.absolute()}")
print(f"Model checkpoint location: {MODELS_DIR.absolute()}")

## 10. Conclusion and Next Steps

### What We Accomplished

In this tutorial, we built a complete deep learning pipeline for medical image classification:

1. ✅ **Data Generation**: Created 2000 synthetic MRI-like tumor images
2. ✅ **Data Pipeline**: Implemented PyTorch datasets with augmentation
3. ✅ **Model Architecture**: Built a CNN with ~1M parameters
4. ✅ **Training**: Trained with validation monitoring and early stopping
5. ✅ **Evaluation**: Comprehensive metrics and clinical analysis
6. ✅ **Visualization**: Interactive dashboards for model interpretation

### Model Performance Summary

Our model achieved:
- **Accuracy**: High overall classification accuracy
- **Sensitivity**: Strong malignant tumor detection
- **Specificity**: Good benign tumor identification
- **ROC-AUC**: Excellent discriminative ability

### Next Steps for Improvement

1. **Real Data**: Replace synthetic data with actual medical imaging datasets
   - Kaggle: Brain Tumor Classification, Breast Cancer Detection
   - NIH Clinical Center datasets
   
2. **Advanced Architectures**:
   - ResNet, DenseNet, EfficientNet
   - Transfer learning from ImageNet
   - Vision Transformers (ViT)
   
3. **Model Interpretation**:
   - Grad-CAM for visualization of important regions
   - SHAP values for feature importance
   - Attention mechanisms
   
4. **Clinical Integration**:
   - Uncertainty quantification
   - Multi-class classification (different tumor types)
   - Tumor segmentation and size estimation
   - Integration with PACS systems
   
5. **Robustness**:
   - Test on different scanner types
   - Validate across demographics
   - Adversarial robustness testing
   - Cross-validation with multiple folds

### Important Ethical Considerations

⚠️ **Medical AI Development Guidelines**:
- Always validate on diverse, representative populations
- Ensure HIPAA compliance and patient privacy
- Obtain proper regulatory approvals (FDA, etc.)
- Use as decision support, not replacement for physicians
- Monitor for bias and regularly retrain
- Maintain human oversight and accountability

### Additional Resources

- **PyTorch Documentation**: https://pytorch.org/docs/
- **Medical Image Analysis**: https://www.sciencedirect.com/journal/medical-image-analysis
- **FDA AI/ML Guidelines**: https://www.fda.gov/medical-devices/software-medical-device-samd/artificial-intelligence-and-machine-learning-aiml-enabled-medical-devices
- **Papers with Code - Medical**: https://paperswithcode.com/area/medical

---

**Thank you for completing this tutorial!** 

You now have a solid foundation for developing deep learning models for medical image classification. Remember to always prioritize patient safety and clinical validation when working with medical AI.